In [0]:
# imports
import mlflow
import mlflow.sklearn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from mlflow.models.signature import infer_signature

In [0]:
# 1) Load a small built-in regression dataset
data = load_diabetes()
diabetes_data = pd.DataFrame(
    data.data,
    columns=data.feature_names
)
diabetes_data.head()


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [0]:
# Partitioning the data

X = diabetes_data
y = pd.Series(
    data.target,
    name="target"
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [0]:
# 2) Build a pipeline: standardize features + linear regression
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

In [0]:
# 3) Create an MLflow run and log parameters, metrics, artifacts, and the model
with mlflow.start_run(run_name="linear-regression-diabetes") as run:
    # ---- Params (useful context even for LinearRegression which has few hyperparams)
    mlflow.log_params({
        "model_type": "LinearRegression",
        "dataset": "sklearn.diabetes",
        "test_size": 0.20,
        "random_state": 42,
        "standardize": True
    })

    # ---- Train
    model.fit(X_train, y_train)

In [0]:
# ---- Predict & metrics
preds = model.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
mae  = float(mean_absolute_error(y_test, preds))
r2   = float(r2_score(y_test, preds))

mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

In [0]:
# Coefficients come from the regressor step inside the pipeline
coefs = pd.DataFrame({
    "feature": X.columns,
    "coef": model.named_steps["regressor"].coef_
})
coefs.to_csv("coefficients.csv", index=False)
mlflow.log_artifact("coefficients.csv", artifact_path="model_info")

In [0]:
 # Residuals plot
residuals = y_test.values - preds
plt.figure(figsize=(6,4))
plt.scatter(preds, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--", linewidth=1)
plt.title("Residuals vs Predictions (Linear Regression)")
plt.xlabel("Predictions")
plt.ylabel("Residuals")
plt.tight_layout()
plt.savefig("residuals.png")
plt.close()

In [0]:
#display the error metrics
print(f"Metrics -> RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

Metrics -> RMSE: 53.8534, MAE: 42.7941, R2: 0.4526
